# LeetCode 551 — Student Attendance Record I bằng Python

**Link đề:** https://leetcode.com/problems/student-attendance-record-i/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def checkRecord(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Đọc đề và tách thành **hai điều kiện độc lập**: số ngày vắng `A` < 2, và không có 3 ngày `L` **liên tiếp**.
2. Nắm các công cụ chuỗi Python: `str.count`, toán tử `in` với chuỗi con, duyệt `for ch in s`, `enumerate`, `itertools.groupby`, `re.search`.
3. Phân biệt "**tổng số lần**" với "**số lần liên tiếp**" — nguồn gốc của rất nhiều lỗi.
4. Viết 4 lời giải: dùng hàm có sẵn → một lần duyệt với bộ đếm → `groupby` → regex; so sánh và chọn cách phù hợp.
5. Trace từng ký tự, chứng minh đúng, test biên và kiểm tra chéo bằng brute force.

## 1. Đề bài

Chuỗi `s` là bảng điểm danh của một học sinh, mỗi ký tự là một ngày:
- `'A'`: Absent (vắng),
- `'L'`: Late (đi muộn),
- `'P'`: Present (có mặt).

Học sinh được **khen thưởng** nếu thỏa **cả hai**:
1. Vắng **ít hơn 2** ngày (tức tối đa 1 ngày `A`) — tính trên **tổng** cả chuỗi.
2. **Không bao giờ** đi muộn **3 ngày liên tiếp trở lên**.

Trả về `True` nếu được khen, ngược lại `False`.

| `s` | Output | Vì sao |
|---|:---:|---|
| `"PPALLP"` | `True` | 1 ngày A, tối đa 2 L liền nhau |
| `"PPALLL"` | `False` | `LLL` ở cuối |
| `"AA"` | `False` | 2 ngày A |
| `"LALL"` | `True` | các L bị A ngăn cách, không có 3 L liền |
| `"PLPLPL"` | `True` | 3 ngày L nhưng **không liên tiếp** |

**Constraints:** `1 <= s.length <= 1000`, `s[i]` là `'A'`, `'L'` hoặc `'P'`.

## 2. Nền tảng Python cần dùng

### 2.1. `str.count(sub)` — đếm số lần xuất hiện

`s.count("A")` trả về số lần ký tự `A` xuất hiện trong `s` (các lần **không chồng lấn**).

In [1]:
s = "PPALLP"
print(s.count("A"))     # 1
print(s.count("L"))     # 2
print("AAA".count("AA"))  # 1 -> đếm không chồng lấn

1
2
1


### 2.2. Toán tử `in` với chuỗi — kiểm tra chuỗi con

`"LLL" in s` trả về `True` nếu `"LLL"` là **chuỗi con liên tiếp** (substring) của `s`. Đây đúng là điều kiện "3 L liên tiếp".

In [2]:
print("LLL" in "PPALLL")   # True
print("LLL" in "PPALLP")   # False
print("LLL" in "LLLL")     # True (4 L liền cũng chứa LLL)
print("LLL" in "LPLPL")    # False (3 L nhưng không liền)

True
False
True
False


### 2.3. Duyệt chuỗi và `enumerate`

`for ch in s` lấy từng ký tự. `for i, ch in enumerate(s)` lấy kèm vị trí `i`.

In [3]:
for i, ch in enumerate("PAL"):
    print(i, ch)

0 P
1 A
2 L


### 2.4. Hai kiểu đếm: tổng và liên tiếp

- **Tổng**: đếm cộng dồn, không bao giờ reset.
- **Liên tiếp** (streak/run): gặp đúng ký tự thì `+1`, gặp ký tự khác thì **reset về 0**.

In [4]:
s = "LLPLLL"
total = 0
streak = 0
for ch in s:
    if ch == "L":
        total += 1
        streak += 1
    else:
        streak = 0
    print(f"ch={ch} | total_L={total} | streak_L={streak}")

ch=L | total_L=1 | streak_L=1
ch=L | total_L=2 | streak_L=2
ch=P | total_L=2 | streak_L=0
ch=L | total_L=3 | streak_L=1
ch=L | total_L=4 | streak_L=2
ch=L | total_L=5 | streak_L=3


Quan sát: `total_L` tăng mãi tới 5, còn `streak_L` bị reset ở `P` rồi đạt 3 ở cuối. Điều kiện 1 của đề cần kiểu **tổng** (cho `A`), điều kiện 2 cần kiểu **liên tiếp** (cho `L`).

## 3. Cách 1 — Dùng hàm có sẵn (ngắn nhất)

Dịch thẳng hai điều kiện:
- Điều kiện 1: `s.count("A") < 2`.
- Điều kiện 2: `"LLL" not in s`.

Kết hợp bằng `and`.

In [5]:
def check_record_builtin(s: str) -> bool:
    return s.count("A") < 2 and "LLL" not in s

for s in ["PPALLP", "PPALLL", "AA", "LALL", "PLPLPL"]:
    print(f"{s!r:>10} -> {check_record_builtin(s)}")

  'PPALLP' -> True
  'PPALLL' -> False
      'AA' -> False
    'LALL' -> True
  'PLPLPL' -> True


### Code nộp LeetCode — Cách 1

In [6]:
class Solution:
    def checkRecord(self, s: str) -> bool:
        return s.count("A") < 2 and "LLL" not in s

print(Solution().checkRecord("PPALLP"), Solution().checkRecord("PPALLL"))

True False


### Độ phức tạp Cách 1

- `s.count("A")`: duyệt chuỗi một lần → `O(n)`.
- `"LLL" in s`: tìm chuỗi con độ dài cố định 3 → `O(n)`.
- Tổng: `O(n)` thời gian, `O(1)` bộ nhớ.

Cách này **đúng và đủ tốt**. Nhưng nó duyệt chuỗi **hai lần**, và che đi cách tư duy "bộ đếm liên tiếp" — kỹ năng cần cho các bài khó hơn (ví dụ LeetCode 552). Vì vậy ta học thêm cách duyệt một lần.

## 4. Nút thắt & insight: một lần duyệt, dừng sớm

Ta muốn:
- Duyệt từng ký tự **một lần**.
- Giữ hai biến: `absent` (tổng số `A`) và `late_streak` (số `L` liên tiếp kết thúc tại vị trí hiện tại).
- **Dừng sớm** (return `False` ngay) khi `absent == 2` hoặc `late_streak == 3` — không cần đọc phần còn lại.

Quy tắc cập nhật cho từng ký tự `ch`:

| `ch` | `absent` | `late_streak` |
|---|---|---|
| `'A'` | `+1` | reset `0` |
| `'L'` | giữ | `+1` |
| `'P'` | giữ | reset `0` |

Chú ý: `'A'` cũng **ngắt** chuỗi `L` liên tiếp (ví dụ `"LALL"`).

## 5. Cách 2 — Một lần duyệt với bộ đếm (lời giải chính)

Viết từng phần:

```python
absent = 0
late_streak = 0
for ch in s:
    if ch == "A":
        absent += 1
        late_streak = 0
        if absent >= 2: return False
    elif ch == "L":
        late_streak += 1
        if late_streak >= 3: return False
    else:  # "P"
        late_streak = 0
return True
```

`elif` = "else if": chỉ kiểm tra khi các nhánh trên sai.

In [7]:
def check_record_scan(s: str) -> bool:
    absent = 0
    late_streak = 0
    for ch in s:
        if ch == "A":
            absent += 1
            late_streak = 0
            if absent >= 2:
                return False
        elif ch == "L":
            late_streak += 1
            if late_streak >= 3:
                return False
        else:
            late_streak = 0
    return True

for s in ["PPALLP", "PPALLL", "AA", "LALL", "PLPLPL"]:
    print(f"{s!r:>10} -> {check_record_scan(s)}")

  'PPALLP' -> True
  'PPALLL' -> False
      'AA' -> False
    'LALL' -> True
  'PLPLPL' -> True


## 6. Trace từng ký tự

Ví dụ `s = "PLLALLLP"`:

| i | ch | absent | late_streak | Quyết định |
|---:|:---:|---:|---:|---|
| 0 | P | 0 | 0 | tiếp |
| 1 | L | 0 | 1 | tiếp |
| 2 | L | 0 | 2 | tiếp |
| 3 | A | 1 | 0 | tiếp (A ngắt chuỗi L) |
| 4 | L | 1 | 1 | tiếp |
| 5 | L | 1 | 2 | tiếp |
| 6 | L | 1 | **3** | **return False** |

Ký tự cuối `P` không bao giờ được đọc nhờ dừng sớm.

In [8]:
def trace_check_record(s: str) -> bool:
    absent = 0
    late_streak = 0
    print(f"s = {s!r}")
    for i, ch in enumerate(s):
        if ch == "A":
            absent += 1
            late_streak = 0
        elif ch == "L":
            late_streak += 1
        else:
            late_streak = 0
        print(f"i={i} ch={ch} | absent={absent} | late_streak={late_streak}")
        if absent >= 2:
            print("-> vắng 2 ngày: return False")
            return False
        if late_streak >= 3:
            print("-> muộn 3 ngày liên tiếp: return False")
            return False
    print("-> duyệt hết, hợp lệ: return True")
    return True

trace_check_record("PLLALLLP")
print()
trace_check_record("PAPLLP")
print()
trace_check_record("LAPA")

s = 'PLLALLLP'
i=0 ch=P | absent=0 | late_streak=0
i=1 ch=L | absent=0 | late_streak=1
i=2 ch=L | absent=0 | late_streak=2
i=3 ch=A | absent=1 | late_streak=0
i=4 ch=L | absent=1 | late_streak=1
i=5 ch=L | absent=1 | late_streak=2
i=6 ch=L | absent=1 | late_streak=3
-> muộn 3 ngày liên tiếp: return False

s = 'PAPLLP'
i=0 ch=P | absent=0 | late_streak=0
i=1 ch=A | absent=1 | late_streak=0
i=2 ch=P | absent=1 | late_streak=0
i=3 ch=L | absent=1 | late_streak=1
i=4 ch=L | absent=1 | late_streak=2
i=5 ch=P | absent=1 | late_streak=0
-> duyệt hết, hợp lệ: return True

s = 'LAPA'
i=0 ch=L | absent=0 | late_streak=1
i=1 ch=A | absent=1 | late_streak=0
i=2 ch=P | absent=1 | late_streak=0
i=3 ch=A | absent=2 | late_streak=0
-> vắng 2 ngày: return False


False

## 7. Code nộp LeetCode — Cách 2

In [9]:
class Solution:
    def checkRecord(self, s: str) -> bool:
        absent = 0
        late_streak = 0
        for ch in s:
            if ch == "A":
                absent += 1
                late_streak = 0
                if absent >= 2:
                    return False
            elif ch == "L":
                late_streak += 1
                if late_streak >= 3:
                    return False
            else:
                late_streak = 0
        return True

sol = Solution()
print(sol.checkRecord("PPALLP"), sol.checkRecord("PPALLL"))

True False


## 8. Cách 3 — Gom nhóm ký tự liên tiếp với `itertools.groupby`

`itertools.groupby(s)` chia chuỗi thành các **nhóm ký tự giống nhau đứng liền nhau** (run). Mỗi phần tử là cặp `(key, iterator_các_phần_tử)`.

Ví dụ `"PPALLL"` → `('P', PP)`, `('A', A)`, `('L', LLL)`.

Điều kiện 2 trở thành: không có nhóm `L` nào dài ≥ 3. Độ dài nhóm tính bằng `len(list(group))` hoặc `sum(1 for _ in group)`.

In [10]:
from itertools import groupby

for key, group in groupby("PPALLLPLL"):
    items = list(group)
    print(key, items, "độ dài", len(items))

P ['P', 'P'] độ dài 2
A ['A'] độ dài 1
L ['L', 'L', 'L'] độ dài 3
P ['P'] độ dài 1
L ['L', 'L'] độ dài 2


In [11]:
def check_record_groupby(s: str) -> bool:
    if s.count("A") >= 2:
        return False
    for key, group in groupby(s):
        if key == "L" and sum(1 for _ in group) >= 3:
            return False
    return True

for s in ["PPALLP", "PPALLL", "AA", "LALL", "PLPLPL"]:
    print(f"{s!r:>10} -> {check_record_groupby(s)}")

  'PPALLP' -> True
  'PPALLL' -> False
      'AA' -> False
    'LALL' -> True
  'PLPLPL' -> True


Lưu ý: `groupby` **chỉ gom các phần tử liền nhau**, không gom toàn bộ chuỗi như `Counter`. Đây chính là thứ ta cần cho "liên tiếp".

`sum(1 for _ in group)`: đếm số phần tử của iterator mà không tạo list; `_` là tên biến quy ước khi ta không dùng giá trị.

## 9. Cách 4 — Biểu thức chính quy (regex)

Module `re` mô tả "mẫu" chuỗi:
- `A.*A`: một `A`, rồi bất kỳ ký tự nào (`.` = một ký tự, `*` = lặp 0 hay nhiều lần), rồi một `A` khác → tức có ≥ 2 `A`.
- `LLL`: ba `L` liền nhau.
- `|` nghĩa là "hoặc".

`re.search(pattern, s)` trả về đối tượng khớp (match) nếu tìm thấy ở đâu đó, ngược lại `None`.

In [12]:
import re

print(re.search(r"A.*A", "PAPPA"))          # khớp
print(re.search(r"A.*A", "PAPP"))           # None
print(re.search(r"A.*A|LLL", "PPALLL"))     # khớp phần LLL

<re.Match object; span=(1, 5), match='APPA'>
None
<re.Match object; span=(3, 6), match='LLL'>


In [13]:
def check_record_regex(s: str) -> bool:
    return re.search(r"A.*A|LLL", s) is None

for s in ["PPALLP", "PPALLL", "AA", "LALL", "PLPLPL"]:
    print(f"{s!r:>10} -> {check_record_regex(s)}")

  'PPALLP' -> True
  'PPALLL' -> False
      'AA' -> False
    'LALL' -> True
  'PLPLPL' -> True


Chuỗi `r"..."` là **raw string**: dấu `\` giữ nguyên, rất tiện khi viết regex. Regex gọn nhưng khó đọc với người mới; trong phỏng vấn nên trình bày Cách 2 trước.

## 10. Vì sao Cách 2 đúng?

**Invariant** sau khi xử lý ký tự thứ `i`:
- `absent` = số `A` trong `s[0..i]`;
- `late_streak` = số `L` liên tiếp **kết thúc tại `i`** (0 nếu `s[i] != 'L'`).

Giữ được invariant: `A` tăng `absent` và ngắt chuỗi L; `L` nối dài chuỗi L; `P` ngắt chuỗi L — đúng định nghĩa.

- Nếu tồn tại đoạn `LLL` kết thúc ở vị trí `j`, thì khi xử lý `j`, `late_streak >= 3` → trả `False` ✔.
- Nếu có ≥ 2 `A`, khi gặp `A` thứ hai thì `absent = 2` → trả `False` ✔.
- Ngược lại, vòng lặp chạy hết mà không vi phạm → cả hai điều kiện thỏa → `True` ✔.

## 11. So sánh các cách và độ phức tạp

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Dừng sớm? | Nhận xét |
|---|---|---|---|:---:|---|
| 1. `count` + `in` | hàm có sẵn | `O(n)` | `O(1)` | không | ngắn nhất, dễ nhớ |
| 2. Một lần duyệt | bộ đếm tổng + liên tiếp | `O(n)` | `O(1)` | **có** | **nền tảng**, mở rộng được |
| 3. `groupby` | gom nhóm liền nhau | `O(n)` | `O(1)`* | một phần | hay cho bài "run" |
| 4. Regex | mẫu `A.*A\|LLL` | `O(n)` thực tế | `O(1)` | có | gọn, khó đọc |

\* `sum(1 for _ in group)` không tạo list. Nếu dùng `len(list(group))` thì cần `O(độ dài nhóm)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, brute force

Ta tạo một **brute force** dịch từng chữ đề bài: đếm A, và thử mọi vị trí `i` xem `s[i], s[i+1], s[i+2]` có cùng là `L` không. Sau đó sinh **mọi chuỗi** độ dài ≤ 7 từ `{A, L, P}` (3 + 9 + ... + 2187 chuỗi) và so 4 cách với brute force.

In [14]:
def check_record_brute(s: str) -> bool:
    absent = sum(1 for ch in s if ch == "A")
    three_late = any(s[i] == s[i + 1] == s[i + 2] == "L" for i in range(len(s) - 2))
    return absent < 2 and not three_late

tests = [
    ("PPALLP", True), ("PPALLL", False), ("A", True), ("L", True), ("P", True),
    ("AA", False), ("LLL", False), ("LL", True), ("LALL", True), ("PLPLPL", True),
    ("ALLAPPL", False), ("LLLA", False), ("PPPPPPPPPPA", True), ("LLPLLPLL", True),
]
methods = [check_record_builtin, check_record_scan, check_record_groupby, check_record_regex, check_record_brute]
for s, expected in tests:
    for f in methods:
        assert f(s) == expected, (f.__name__, s)
    print(f"{s!r:>14} -> {expected}")
print("Test mẫu PASS!")

      'PPALLP' -> True
      'PPALLL' -> False
           'A' -> True
           'L' -> True
           'P' -> True
          'AA' -> False
         'LLL' -> False
          'LL' -> True
        'LALL' -> True
      'PLPLPL' -> True
     'ALLAPPL' -> False
        'LLLA' -> False
 'PPPPPPPPPPA' -> True
    'LLPLLPLL' -> True
Test mẫu PASS!


`any(...)` trả `True` nếu **ít nhất một** phần tử là `True`. `a == b == c == "L"` là **so sánh chuỗi** (chained comparison) của Python, tương đương `a == b and b == c and c == "L"`.

`itertools.product("ALP", repeat=k)` sinh mọi bộ `k` ký tự — dùng để kiểm tra vét cạn.

In [15]:
from itertools import product

count = 0
for length in range(1, 8):
    for tup in product("ALP", repeat=length):
        s = "".join(tup)
        expected = check_record_brute(s)
        for f in methods:
            assert f(s) == expected, (f.__name__, s)
        count += 1
print(f"Đã kiểm tra vét cạn {count} chuỗi: mọi cách khớp brute force.")

Đã kiểm tra vét cạn 3279 chuỗi: mọi cách khớp brute force.


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đếm **tổng** số L thay vì liên tiếp

In [16]:
def wrong_total_late(s: str) -> bool:
    return s.count("A") < 2 and s.count("L") < 3    # SAI

print("PLPLPL ->", wrong_total_late("PLPLPL"), "| đúng:", check_record_scan("PLPLPL"))

PLPLPL -> False | đúng: True


Vì sao sai: đề cấm 3 L **liên tiếp**, không cấm 3 L tổng cộng. Sửa: `"LLL" not in s` hoặc dùng bộ đếm có reset.

### Lỗi 2: Quên reset chuỗi L khi gặp `'A'`

Chỉ reset khi gặp `P`, quên rằng `A` cũng ngắt chuỗi L.

In [17]:
def wrong_no_reset_on_A(s: str) -> bool:
    absent = late_streak = 0
    for ch in s:
        if ch == "A":
            absent += 1               # quên late_streak = 0
        elif ch == "L":
            late_streak += 1
        else:
            late_streak = 0
        if absent >= 2 or late_streak >= 3:
            return False
    return True

print("LALL ->", wrong_no_reset_on_A("LALL"), "| đúng:", check_record_scan("LALL"))

LALL -> False | đúng: True


### Lỗi 3: Dùng `<= 2` thay vì `< 2` cho số ngày vắng

"Strictly fewer than 2" nghĩa là **nhỏ hơn hẳn** 2 → tối đa 1. Viết `s.count("A") <= 2` sẽ cho `"AA"` là `True` — sai.

### Lỗi 4: Reset bộ đếm **vắng** khi gặp `P`

`absent` là tổng cả chuỗi, **không bao giờ** reset. Chỉ `late_streak` mới reset.

In [18]:
def wrong_reset_absent(s: str) -> bool:
    absent = late_streak = 0
    for ch in s:
        if ch == "A":
            absent += 1
            late_streak = 0
        elif ch == "L":
            late_streak += 1
        else:
            late_streak = 0
            absent = 0                # SAI
        if absent >= 2 or late_streak >= 3:
            return False
    return True

print("APA ->", wrong_reset_absent("APA"), "| đúng:", check_record_scan("APA"))

APA -> True | đúng: False


### Lỗi 5: Viết `if ch == "A" or "L":`

Python hiểu là `(ch == "A") or ("L")`, mà chuỗi khác rỗng luôn "truthy" → điều kiện luôn đúng. Phải viết `ch == "A" or ch == "L"` hoặc `ch in "AL"`.

In [19]:
ch = "P"
print(bool(ch == "A" or "L"))       # True  (sai ý định!)
print(ch == "A" or ch == "L")       # False
print(ch in ("A", "L"))             # False

True
False
False


## 14. Mẹo nhận dạng pattern và biến thể

### Mẹo
- "**Tổng cộng** ít hơn k lần" → bộ đếm tổng / `count`.
- "**Liên tiếp** k lần" → bộ đếm streak có reset, hoặc kiểm tra chuỗi con `"X" * k in s`, hoặc `groupby`.
- Chỉ cần trả lời đúng/sai → **dừng sớm** ngay khi vi phạm.

### Biến thể
1. **LeetCode 552 – Student Attendance Record II** (Hard): đếm số chuỗi độ dài `n` được khen — dùng DP với trạng thái `(absent, late_streak)` chính là hai biến của Cách 2.
2. **LeetCode 485 – Max Consecutive Ones**: độ dài streak dài nhất.
3. **LeetCode 1446 – Consecutive Characters**: run dài nhất của cùng một ký tự.
4. Tổng quát hóa: cho ngưỡng `max_absent` và `max_late_streak` tùy ý.

In [20]:
# Biến thể 1 (xem trước ý tưởng): đếm số chuỗi độ dài n được khen bằng DP trên trạng thái (absent, late_streak)
def count_rewardable(n: int) -> int:
    dp = {(0, 0): 1}                    # trạng thái -> số chuỗi
    for _ in range(n):
        new = {}
        for (a, l), ways in dp.items():
            for nxt in ((a, 0), (a + 1, 0), (a, l + 1)):   # thêm P, A, L
                if nxt[0] < 2 and nxt[1] < 3:
                    new[nxt] = new.get(nxt, 0) + ways
        dp = new
    return sum(dp.values())

print([count_rewardable(n) for n in range(1, 6)])   # n=2 -> 8 theo đề 552
# đối chiếu vét cạn
print([sum(check_record_scan("".join(t)) for t in product("ALP", repeat=n)) for n in range(1, 6)])

[3, 8, 19, 43, 94]
[3, 8, 19, 43, 94]


Hai dòng in ra trùng nhau: DP đếm đúng. `dict.items()` trả về các cặp `(key, value)`; `new.get(nxt, 0) + ways` cộng dồn vào trạng thái mới.

## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `count_char(s, c)` đếm số lần ký tự `c` xuất hiện **không dùng** `str.count`.

In [21]:
def count_char(s: str, c: str) -> int:
    # TODO
    pass

# assert count_char("PPALLP", "L") == 2
# assert count_char("PPP", "A") == 0

### Bài 2 (dễ)
Viết `longest_run(s, c)`: độ dài đoạn liên tiếp dài nhất chỉ gồm ký tự `c`.

In [22]:
def longest_run(s: str, c: str) -> int:
    # TODO
    pass

# assert longest_run("LLPLLLP", "L") == 3
# assert longest_run("PPP", "L") == 0

### Bài 3 (vừa)
Viết `check_record_general(s, max_absent, max_late)`: được khen nếu số `A` **< `max_absent`** và không có `max_late` ngày `L` liên tiếp. (Bài gốc: `max_absent=2, max_late=3`.)

In [23]:
def check_record_general(s: str, max_absent: int, max_late: int) -> bool:
    # TODO
    pass

# assert check_record_general("PPALLP", 2, 3) == True
# assert check_record_general("PPALLP", 2, 2) == False
# assert check_record_general("AAP", 3, 3) == True

### Bài 4 (vừa)
Viết `first_violation(s)`: trả về **chỉ số** đầu tiên mà tại đó học sinh mất quyền khen thưởng, hoặc `-1` nếu không vi phạm.

In [24]:
def first_violation(s: str) -> int:
    # TODO
    pass

# assert first_violation("PLLALLLP") == 6
# assert first_violation("APA") == 2
# assert first_violation("PPALLP") == -1

### Bài 5 (khó hơn)
Có danh sách nhiều học sinh `records` (list chuỗi). Trả về list **chỉ số** các học sinh được khen, sắp theo số ngày `P` giảm dần (bằng nhau thì chỉ số tăng dần).

In [25]:
def rewarded_ranking(records: list[str]) -> list[int]:
    # TODO
    pass

# assert rewarded_ranking(["PPALLP", "PPALLL", "PPPP", "AA", "LP"]) == [2, 0, 4]

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Duyệt `for ch in s` | Biến đếm bắt đầu 0 | `if ch == c: cnt += 1` |
| 2 | Hai biến `cur`, `best` | Gặp `c` thì `cur += 1`, khác thì `cur = 0` | Cập nhật `best = max(best, cur)` mỗi bước |
| 3 | Giống Cách 2 | Thay số 2, 3 bằng tham số | `absent >= max_absent` / `late >= max_late` → False |
| 4 | Giống trace | Trả `i` thay vì `False` | Duyệt hết thì trả `-1` |
| 5 | Lọc bằng `check_record_scan` | Ghép cặp `(index, count P)` | `sorted(..., key=lambda x: (-p, i))` |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [26]:
def answer_count_char(s: str, c: str) -> int:
    cnt = 0
    for ch in s:
        if ch == c:
            cnt += 1
    return cnt

def answer_longest_run(s: str, c: str) -> int:
    cur = best = 0
    for ch in s:
        cur = cur + 1 if ch == c else 0
        best = max(best, cur)
    return best

def answer_check_record_general(s: str, max_absent: int, max_late: int) -> bool:
    absent = late = 0
    for ch in s:
        if ch == "A":
            absent += 1
            late = 0
        elif ch == "L":
            late += 1
        else:
            late = 0
        if absent >= max_absent or late >= max_late:
            return False
    return True

def answer_first_violation(s: str) -> int:
    absent = late = 0
    for i, ch in enumerate(s):
        if ch == "A":
            absent += 1
            late = 0
        elif ch == "L":
            late += 1
        else:
            late = 0
        if absent >= 2 or late >= 3:
            return i
    return -1

def answer_rewarded_ranking(records: list[str]) -> list[int]:
    ok = [(i, r.count("P")) for i, r in enumerate(records) if check_record_scan(r)]
    ok.sort(key=lambda item: (-item[1], item[0]))
    return [i for i, _ in ok]

assert answer_count_char("PPALLP", "L") == 2
assert answer_count_char("PPP", "A") == 0
assert answer_longest_run("LLPLLLP", "L") == 3
assert answer_longest_run("PPP", "L") == 0
assert answer_check_record_general("PPALLP", 2, 3) == True
assert answer_check_record_general("PPALLP", 2, 2) == False
assert answer_check_record_general("AAP", 3, 3) == True
assert answer_first_violation("PLLALLLP") == 6
assert answer_first_violation("APA") == 2
assert answer_first_violation("PPALLP") == -1
assert answer_rewarded_ranking(["PPALLP", "PPALLL", "PPPP", "AA", "LP"]) == [2, 0, 4]
for length in range(1, 7):
    for t in product("ALP", repeat=length):
        s = "".join(t)
        assert answer_check_record_general(s, 2, 3) == check_record_scan(s)
        assert (answer_first_violation(s) == -1) == check_record_scan(s)
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


`lambda item: (-item[1], item[0])` là **hàm ẩn danh** trả về tuple khóa sắp xếp: số `P` âm (để giảm dần), rồi chỉ số (tăng dần). Python so sánh tuple theo thứ tự từng phần tử.

## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def checkRecord(self, s: str) -> bool:
        return s.count("A") < 2 and "LLL" not in s
```

Bản một lần duyệt (dùng khi cần mở rộng):

```python
absent = late = 0
for ch in s:
    if ch == "A":
        absent += 1; late = 0
    elif ch == "L":
        late += 1
    else:
        late = 0
    if absent >= 2 or late >= 3:
        return False
return True
```

### Checklist trước khi submit
- [ ] Điều kiện vắng là `< 2` (nghiêm ngặt), tính trên **tổng**.
- [ ] Điều kiện muộn là **liên tiếp** 3, không phải tổng 3.
- [ ] `A` và `P` đều reset chuỗi L; **không** reset `absent`.
- [ ] Không viết `ch == "A" or "L"`.
- [ ] Đã thử `"LALL"`, `"PLPLPL"`, `"AA"`, `"LLL"`, chuỗi 1 ký tự.